# MOSS-Image · T4 후보 비교
[최신 Colab 노트북](https://github.com/HisameOgasahara/tmp_moss#readme)

**1 → 2 → 3 → 4 → 5 실행.** 3번 셀에서 기본 후보·단일 옵션·조합과 측정 조건을 함께 선택합니다.

- 개별 검사: 로딩·인코딩·각 시험 연산·VAE 시간을 각각 기록합니다.
- 첫 생성: 새 프로세스 시작 및 로딩 시작부터 첫 PNG 저장 완료까지 각각 측정합니다.
- 모델 유지: 같은 프로세스에서 시드 변경 → 프롬프트 변경 → 스텝·CFG 변경 순으로 재생성합니다. 인코딩 캐시 사용 여부도 기록합니다.

첫 생성은 파일 다운로드가 완료된 상태에서 측정합니다. 기본 후보는 50스텝·CFG 3, LoRA 후보는 8스텝·CFG 1이며 3번 셀에서 변경할 수 있습니다.


In [ ]:
#@title 1. T4 환경 확인
import sys, subprocess, json, os, time, shutil, hashlib, urllib.request
from pathlib import Path
from IPython.display import display, clear_output

ROOT = Path('/content/moss_t4')
ROOT.mkdir(parents=True, exist_ok=True)
gpu = subprocess.check_output([
    'nvidia-smi', '--query-gpu=name,memory.total,driver_version', '--format=csv,noheader'
], text=True).strip()
if 'T4' not in gpu:
    raise RuntimeError(f'T4 GPU를 선택하세요. 현재 GPU: {gpu}')
print('GPU / VRAM / driver:', gpu)
print('RAM:', round(os.sysconf('SC_PAGE_SIZE') * os.sysconf('SC_PHYS_PAGES') / 2**30, 2), 'GiB')
print('디스크 여유:', round(shutil.disk_usage(ROOT).free / 2**30, 2), 'GiB')
subprocess.run([sys.executable, '-c',
    'import torch; print("torch:", torch.__version__, "CUDA:", torch.version.cuda, "SM:", torch.cuda.get_device_capability())'], check=True)


In [ ]:
#@title 2. ComfyUI 설치
COMFY_REV = 'bc5f880dbdd43e426077f42ccec53bb577ffee9f'
MODEL_REV = '24147db865fcdc207270f844777587223fed8d6e'
FIXED_REV = 'ba6360149256790c5434e4a5b8ebf2c9899785b4'
SPECTRUM_REV = 'd3f7e243ea2db6cc3ff26f42b935b167f938b599'
COMFY = ROOT / 'ComfyUI'
VENV = ROOT / 'venv'
PY = str(VENV / 'bin/python')

INSTALL_LOG = ROOT / 'install.log'
INSTALL_LOG.write_text('', encoding='utf-8')

def run(command, **kwargs):
    command = [str(x) for x in command]
    lines = []
    with INSTALL_LOG.open('a', encoding='utf-8') as log:
        log.write('> ' + ' '.join(command) + chr(10))
        with subprocess.Popen(command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                              text=True, bufsize=1, **kwargs) as process:
            for line in process.stdout:
                print(line, end='', flush=True)
                log.write(line)
                log.flush()
                lines.append(line)
            code = process.wait()
    if code:
        tail = ''.join(lines[-40:])
        raise RuntimeError(f'설치 명령 실패 (exit {code}) · 로그: {INSTALL_LOG}' + chr(10) + tail)

def fetch(url):
    request = urllib.request.Request(url, headers={'User-Agent': 'moss-t4-smoke'})
    with urllib.request.urlopen(request, timeout=60) as response:
        return response.read()

if shutil.which('uv') is None:
    run([sys.executable, '-m', 'pip', 'install', '-q', 'uv>=0.9.27'])
uv_help = subprocess.check_output(['uv', 'pip', 'install', '--help'], text=True)
if '--excludes' not in uv_help:
    run([sys.executable, '-m', 'pip', 'install', '-q', '--upgrade', 'uv>=0.9.27'])
if not Path(PY).exists():
    run(['uv', 'venv', '--system-site-packages', '--python', sys.executable, VENV])
if not (COMFY / '.git').exists():
    run(['git', 'clone', '--quiet', 'https://github.com/Comfy-Org/ComfyUI.git', COMFY])
run(['git', '-C', COMFY, 'checkout', '--quiet', COMFY_REV])
# Colab의 torch/CUDA 패키지는 상속하고 uv의 재설치 대상에서 제외합니다.
runtime_code = """
import importlib.metadata as m, json
packages = {}
for dist in m.distributions():
    name = dist.metadata['Name'].lower().replace('_', '-')
    if name in ('torch', 'torchvision', 'torchaudio', 'triton') or name.startswith('nvidia-'):
        packages[name] = dist.version
print(json.dumps(packages))
"""
runtime_packages = json.loads(subprocess.check_output(
    [sys.executable, '-c', runtime_code], text=True))
excludes = ROOT / 'colab-runtime-excludes.txt'
excludes.write_text(chr(10).join(sorted(runtime_packages)) + chr(10), encoding='utf-8')
print('Colab 런타임 유지:', runtime_packages)
run(['uv', 'pip', 'install', '--python', PY, '--excludes', excludes,
     '-r', COMFY / 'requirements.txt', 'huggingface_hub', 'psutil', 'pandas'])

source = json.loads(fetch('https://api.github.com/repos/HisameOgasahara/tmp_moss/commits/main'))['sha']
worker = ROOT / 'moss_smoke_worker.py'
worker.write_bytes(fetch(f'https://raw.githubusercontent.com/HisameOgasahara/tmp_moss/{source}/moss_smoke_worker.py'))
environment = {
    'source_revision': source, 'comfy_revision': COMFY_REV, 'model_revision': MODEL_REV,
    'fixed_revision': FIXED_REV, 'spectrum_revision': SPECTRUM_REV, 'gpu': gpu,
    'colab_runtime_packages': runtime_packages,
    'worker_sha256': hashlib.sha256(worker.read_bytes()).hexdigest(),
    'packages': subprocess.check_output([PY, '-m', 'pip', 'freeze'], text=True) if (VENV / 'bin/pip').exists()
        else subprocess.check_output(['uv', 'pip', 'freeze', '--python', PY], text=True),
}
(ROOT / 'environment.json').write_text(json.dumps(environment, indent=2), encoding='utf-8')
print('설치 완료:', COMFY_REV[:12])


| 기본 ID | 본체 / 인코더 | 본체 연산 설정 |
|---|---|---|
| N1 | INT8 / W4A8 | 자동 |
| N2 | INT8 / INT8 | 자동 |
| N3 | INT8 / INT8 | FP16 옵션 |
| N4 | FP16-fixed / INT8 | FP16 옵션 |

옵션: `lora8`, `spectrum`, `compile`, `pytorch_attention`, `tiled_vae`, `lowvram`, `fp8`.
조합 예: `N1:lora8;N1:lora8+spectrum`. 후보마다 동일한 첫 프롬프트·변경 프롬프트·시드·해상도를 적용합니다.


In [ ]:
#@title 3. 후보·조합·실사용 조건 선택
BASE_IDS = '' #@param {type:'string'}
OPTION_BASE = 'N3' #@param ['N1','N2','N3','N4']
OPTIONS = 'lora8' #@param {type:'string'}
COMBINATIONS = 'N1:lora8;N1:lora8+spectrum' #@param {type:'string'}
RUN_SMOKE = True #@param {type:'boolean'}
RUN_BENCHMARK = True #@param {type:'boolean'}
REUSE_PASSED_SMOKE = True #@param {type:'boolean'}
WIDTH = 1024 #@param {type:'integer'}
HEIGHT = 1024 #@param {type:'integer'}
PROMPT = 'A studio photograph of a red ceramic teapot on a wooden table, soft window light.' #@param {type:'string'}
WARM_PROMPT = 'A studio photograph of a blue ceramic teapot on a wooden table, soft window light.' #@param {type:'string'}
SEED = 42 #@param {type:'integer'}
BASE_STEPS = 50 #@param {type:'integer'}
BASE_CFG = 3.0 #@param {type:'number'}
LORA_STEPS = 8 #@param {type:'integer'}
LORA_CFG = 1.0 #@param {type:'number'}
WARM_STEP_DELTA = 2 #@param {type:'integer'}
WARM_CFG_DELTA = 0.0 #@param {type:'number'}
LOAD_TIMEOUT = 120 #@param {type:'integer'}
ENCODE_TIMEOUT = 60 #@param {type:'integer'}
FORWARD_TIMEOUT = 60 #@param {type:'integer'}
VAE_TIMEOUT = 30 #@param {type:'integer'}
COMPILE_TIMEOUT = 180 #@param {type:'integer'}
GENERATION_TIMEOUT = 1800 #@param {type:'integer'}

# BASE_IDS: 기본 후보. OPTIONS: OPTION_BASE에 각각 적용. COMBINATIONS: 세미콜론으로 구분.
import copy
if WIDTH <= 0 or HEIGHT <= 0 or WIDTH % 16 or HEIGHT % 16:
    raise ValueError('가로·세로는 양의 16배수여야 합니다.')
base = dict(dit='moss_image_sft_int8_convrot.safetensors', te='qwen3.5_9b_int8_convrot.safetensors',
            vae='flux2-vae_small.safetensors', flags=[], options=[], weight_dtype='default',
            comfy_root=str(COMFY), width=WIDTH, height=HEIGHT, prompt=PROMPT, warm_prompt=WARM_PROMPT, seed=SEED)
registry = {key: copy.deepcopy(base) for key in ['N1','N2','N3','N4']}
registry['N1']['te'] = 'qwen3.5_9b_w4a8_gptq.safetensors'
registry['N3']['flags'] = ['--fp16-unet']
registry['N4'].update(dit='moss_image_sft_fp16_fixed.safetensors', flags=['--fp16-unet'])
valid_options = {'lora8','spectrum','compile','pytorch_attention','tiled_vae','lowvram','fp8'}
candidates = []

def add_candidate(base_id, options):
    options = list(dict.fromkeys(options))
    if base_id not in registry or set(options)-valid_options:
        raise ValueError(f'후보 또는 옵션 오류: {base_id}: {options}')
    item = copy.deepcopy(registry[base_id])
    item.update(id=base_id + ('-' + '+'.join(options) if options else ''), options=options)
    if 'pytorch_attention' in options:
        item['flags'].append('--use-pytorch-cross-attention')
    if 'lowvram' in options:
        item['flags'].extend(['--disable-dynamic-vram','--lowvram'])
    if 'fp8' in options:
        item.update(dit='moss_image_sft_bf16.safetensors', weight_dtype='fp8_e4m3fn')
    if 'lora8' in options:
        item.update(lora='moss_sft_8step_lora.safetensors', lora_strength=1.0)
    if 'spectrum' in options:
        item.update(spectrum_path=str(ROOT/'spectrum'/'__init__.py'), spectrum_calls=5,
                    spectrum_config={'mode':'manual','warmup_steps':2,'tail_actual_steps':0,
                                     'history_points':2,'chebyshev_degree':1})
    item.update(steps=LORA_STEPS if 'lora8' in options else BASE_STEPS,
                cfg=LORA_CFG if 'lora8' in options else BASE_CFG)
    item.update(warm_steps=item['steps']+WARM_STEP_DELTA, warm_cfg=item['cfg']+WARM_CFG_DELTA)
    if item['steps'] < 1 or item['warm_steps'] < 1:
        raise ValueError('스텝 수는 1 이상이어야 합니다.')
    if not any(c['id']==item['id'] for c in candidates):
        candidates.append(item)

for key in filter(None, (x.strip() for x in BASE_IDS.split(','))):
    add_candidate(key, [])
for option in filter(None, (x.strip() for x in OPTIONS.split(','))):
    add_candidate(OPTION_BASE, [option])
for combination in filter(None, (x.strip() for x in COMBINATIONS.split(';'))):
    key, options = combination.split(':', 1)
    add_candidate(key.strip(), [x.strip() for x in options.split('+') if x.strip()])
if not candidates:
    raise ValueError('후보를 하나 이상 선택하세요.')
selected_options = sorted({option for item in candidates for option in item['options']})
source = json.loads(fetch('https://api.github.com/repos/HisameOgasahara/tmp_moss/commits/main'))['sha']
worker = ROOT/'moss_smoke_worker.py'
worker.write_bytes(fetch(f'https://raw.githubusercontent.com/HisameOgasahara/tmp_moss/{source}/moss_smoke_worker.py'))
environment.update(source_revision=source, worker_sha256=hashlib.sha256(worker.read_bytes()).hexdigest())
(ROOT/'environment.json').write_text(json.dumps(environment, indent=2), encoding='utf-8')
display(__import__('pandas').DataFrame([{'후보':c['id'], '첫 생성 스텝':c['steps'], 'CFG':c['cfg'],
    '설정 변경 스텝':c['warm_steps'], '설정 변경 CFG':c['warm_cfg']} for c in candidates]))


In [ ]:
#@title 4. 선택한 파일 다운로드
resources = {}
for candidate in candidates:
    for folder, key in [('diffusion_models','dit'), ('text_encoders','te'), ('vae','vae')]:
        name = candidate[key]
        fixed = name == 'moss_image_sft_fp16_fixed.safetensors'
        resources[(folder, name)] = {
            'repo': 'KBlueLeaf/MOSS-Image-FP16-fixed' if fixed else 'Comfy-Org/MOSS-Image',
            'revision': FIXED_REV if fixed else MODEL_REV,
            'filename': f'comfyui/{name}' if fixed else f'{folder}/{name}',
            'target': str(COMFY / 'models' / folder / name),
        }
if 'lora8' in selected_options:
    lora_repo = 'lodestones/MOSS-Image-SFT-8-steps-dmd'
    lora_rev = json.loads(fetch(f'https://huggingface.co/api/models/{lora_repo}'))['sha']
    resources[('loras', 'moss_sft_8step_lora.safetensors')] = {
        'repo': lora_repo, 'revision': lora_rev, 'filename': 'lora.safetensors',
        'target': str(COMFY / 'models/loras/moss_sft_8step_lora.safetensors'),
    }
manifest = ROOT / 'downloads.json'
manifest.write_text(json.dumps(list(resources.values()), indent=2), encoding='utf-8')
download_code = """
import json, os, sys
from pathlib import Path
from huggingface_hub import hf_hub_download
for item in json.loads(Path(sys.argv[1]).read_text()):
    print('Download:', item['filename'], flush=True)
    source = hf_hub_download(item['repo'], item['filename'], revision=item['revision'], cache_dir=sys.argv[2])
    target = Path(item['target'])
    target.parent.mkdir(parents=True, exist_ok=True)
    if target.is_symlink():
        target.unlink()
    if not target.exists():
        target.symlink_to(source)
"""
run([PY, '-c', download_code, manifest, ROOT / 'hf-cache'])
if 'spectrum' in selected_options:
    addon = ROOT / 'spectrum'
    addon.mkdir(exist_ok=True)
    for filename in ['__init__.py', 'nodes.py', 'er_sde_bridge.py']:
        addon.joinpath(filename).write_bytes(fetch(
            f'https://raw.githubusercontent.com/Rkkss/ComfyUI-Spectrum-MOSS/{SPECTRUM_REV}/{filename}'))
print('다운로드 완료')


In [ ]:
#@title 5. 개별 검사·첫 생성·모델 유지 재생성 측정
import signal, zipfile
import pandas as pd
import psutil

RESULTS = ROOT/'results'/'v2'
RESULTS.mkdir(parents=True, exist_ok=True)

def read_state(path):
    try:
        return json.loads(path.read_text(encoding='utf-8'))
    except (OSError, ValueError):
        return {}

def cgroup_oom_count():
    try:
        return int(dict(line.split() for line in Path('/sys/fs/cgroup/memory.events').read_text().splitlines()).get('oom_kill',0))
    except (OSError, ValueError):
        return 0

def stage_limit(stage, item):
    if stage.endswith('.sample'):
        return GENERATION_TIMEOUT
    if 'encode_' in stage:
        return ENCODE_TIMEOUT
    if 'decode_vae' in stage:
        return VAE_TIMEOUT
    if '.forward' in stage:
        return COMPILE_TIMEOUT if 'compile' in item['options'] else FORWARD_TIMEOUT
    return LOAD_TIMEOUT

def smoke_key(item):
    keys = ('id','dit','te','vae','flags','options','weight_dtype','width','height','prompt','seed',
            'lora','lora_strength','spectrum_calls','spectrum_config')
    payload = {'config':{k:item[k] for k in keys if k in item}, 'worker':environment['worker_sha256'],
               'comfy':COMFY_REV, 'model':MODEL_REV, 'fixed':FIXED_REV, 'spectrum':SPECTRUM_REV,
               'runtime':environment.get('colab_runtime_packages'), 'packages':environment.get('packages'),
               'gpu':environment.get('gpu')}
    return hashlib.sha256(json.dumps(payload,sort_keys=True).encode()).hexdigest()

def run_one(item, mode):
    tag = item['id'] + '.' + mode
    state_path, result_path = RESULTS/f'{tag}.state.json', RESULTS/f'{tag}.json'
    task = dict(copy.deepcopy(item), mode=mode, state_path=str(state_path), result_path=str(result_path), output_dir=str(RESULTS))
    task_path = RESULTS/f'{tag}.config.json'
    task_path.write_text(json.dumps(task,ensure_ascii=False,indent=2),encoding='utf-8')
    for path in (state_path,result_path):
        path.unlink(missing_ok=True)
    before_oom = cgroup_oom_count()
    peaks = {}
    last_stage = None
    external_status = None
    with (RESULTS/f'{tag}.log').open('w') as log:
        launched = time.monotonic()
        process = subprocess.Popen([PY,'-u',worker,task_path],cwd=COMFY,
            stdout=log,stderr=subprocess.STDOUT,start_new_session=True)
        try:
            while process.poll() is None:
                state = read_state(state_path)
                stage = state.get('stage','import')
                if stage != last_stage:
                    print(f'{tag}: {stage}',flush=True)
                    last_stage = stage
                if time.monotonic()-state.get('stage_started',launched)>stage_limit(stage,item):
                    external_status = 'TIMEOUT'
                    break
                ram = 0
                try:
                    parent = psutil.Process(process.pid)
                    for child in [parent]+parent.children(recursive=True):
                        try:
                            ram += child.memory_info().rss
                        except psutil.Error:
                            pass
                except psutil.Error:
                    pass
                gpu_used = 0
                try:
                    gpu_used = float(subprocess.check_output(['nvidia-smi','--query-gpu=memory.used',
                        '--format=csv,noheader,nounits'],text=True,timeout=3).splitlines()[0])/1024
                except (subprocess.SubprocessError,ValueError):
                    pass
                active = state.get('active_request','first' if mode=='benchmark' else 'setup')
                for key in ('whole_process',active):
                    peak = peaks.setdefault(key,dict(host_rss_gib=0,gpu_used_gib=0))
                    peak['host_rss_gib'] = max(peak['host_rss_gib'],ram/2**30)
                    peak['gpu_used_gib'] = max(peak['gpu_used_gib'],gpu_used)
                time.sleep(0.5)
        finally:
            if process.poll() is None:
                os.killpg(process.pid,signal.SIGKILL)
            process.wait()
    result = read_state(result_path) or read_state(state_path) or dict(candidate=item['id'],stage='import',config=task)
    if external_status:
        result.update(status=external_status,error='단계별 제한시간 초과')
    elif process.returncode != 0:
        if cgroup_oom_count()>before_oom:
            result.update(status='HOST_OOM',error='cgroup oom_kill 증가')
        elif result.get('status') in (None,'RUNNING') or result.get('status','').startswith('PASS_'):
            result.update(status='PROCESS_EXIT',error=f'프로세스 종료 코드 {process.returncode}')
    for row in result.get('scenarios',[]):
        if row['status']=='RUNNING':
            row.update(status=result.get('status','PROCESS_EXIT'),error=result.get('error',''))
    result.update(mode=mode,process_wall_s=round(time.monotonic()-launched,6),exit_code=process.returncode,
                  resource_peaks={k:{a:round(b,3) for a,b in v.items()} for k,v in peaks.items()},smoke_key=smoke_key(item))
    if result.get('first_image_ready_at'):
        result['process_to_first_image_s'] = round(result['first_image_ready_at']-launched,6)
    result_path.write_text(json.dumps(result,ensure_ascii=False,indent=2),encoding='utf-8')
    return result

def show_results(smokes, benchmarks):
    smoke_rows = []
    for result in smokes:
        for event in result.get('events',[]):
            smoke_rows.append({'후보':result['candidate'],'검사':event['stage'],'결과':event['status'],
                               '개별 초':event['seconds'],'연산 경로':event.get('execution'),
                               '조건 분기':event.get('branch'),'sigma':event.get('sigma')})
    frame = pd.DataFrame(smoke_rows)
    frame.to_csv(RESULTS/'smoke_stages.csv',index=False,encoding='utf-8-sig')
    print('개별 검사 시간')
    display(frame)
    display(pd.DataFrame([{'후보':r['candidate'],'스모크 결과':r['status'],'실패 단계':r.get('stage'),
                          '시험 호출 수':r.get('forward_calls'),'오류':r.get('error','')} for r in smokes]))
    names = {'first':'첫 생성','warm_same_prompt':'모델 유지·시드 변경',
             'warm_new_prompt':'모델 유지·프롬프트 변경','warm_settings':'모델 유지·설정 변경'}
    rows = []
    for result in benchmarks:
        for row in result.get('scenarios',[]):
            times = row.get('stages_s',{})
            first = row['name']=='first'
            peak = result.get('resource_peaks',{}).get(row['name'],{})
            rows.append({'후보':result['candidate'],'실사용':names[row['name']],'결과':row['status'],
                '스텝':row['steps'],'CFG':row['cfg'],'시드':row['seed'],'해상도':f"{result['config']['width']}x{result['config']['height']}",
                '프로세스 시작→첫 이미지 초':result.get('process_to_first_image_s') if first else None,
                '로딩 시작→첫 이미지 초':result.get('load_to_first_image_s') if first else None,
                '요청→이미지 초':row.get('request_s'),
                '양성 인코딩 초':times.get('encode_positive'),'음성 인코딩 초':times.get('encode_negative'),
                '인코딩 캐시':row.get('positive_cache_hit'),'토큰':row.get('positive_tokens'),
                '샘플링·전송 초':times.get('sample'),'VAE 초':times.get('decode_vae'),'저장 초':times.get('save_image'),
                '기타 요청 초':row.get('other_request_s'),'VRAM GiB':peak.get('gpu_used_gib'),'RAM GiB':peak.get('host_rss_gib'),
                'VAE 경로':row.get('vae_actual_path'),'본체 dtype':row.get('dit_compute_dtype')})
    frame = pd.DataFrame(rows)
    frame.to_csv(RESULTS/'usage.csv',index=False,encoding='utf-8-sig')
    print('실제 이미지 생성 시간')
    display(frame)
    display(pd.DataFrame([{'후보':r['candidate'],'실사용 결과':r['status'],'실패 단계':r.get('stage'),
                          '오류':r.get('error','')} for r in benchmarks]))
    summary = dict(schema_version=2,smoke=smokes,benchmark=benchmarks)
    (RESULTS/'summary.json').write_text(json.dumps(summary,ensure_ascii=False,indent=2),encoding='utf-8')

smoke_results, benchmark_results = [], []
for candidate in candidates:
    if RUN_SMOKE:
        cached = read_state(RESULTS/f"{candidate['id']}.smoke.json") if REUSE_PASSED_SMOKE else {}
        if cached.get('status')=='PASS_SMOKE' and cached.get('smoke_key')==smoke_key(candidate):
            print(candidate['id'],': 기존 스모크 통과 결과 재사용')
            smoke_result = cached
        else:
            smoke_result = run_one(candidate,'smoke')
        smoke_results.append(smoke_result)
        if smoke_result['status']!='PASS_SMOKE':
            show_results(smoke_results,benchmark_results)
            continue
    if RUN_BENCHMARK:
        # 별도 새 프로세스에서 첫 생성 후 모델 객체를 유지하며 3번 재생성합니다.
        benchmark_results.append(run_one(candidate,'benchmark'))
    show_results(smoke_results,benchmark_results)


In [ ]:
#@title 6. 개별 검사·실사용 결과·이미지 다운로드
from google.colab import files
archive = ROOT/'moss_t4_results.zip'
with zipfile.ZipFile(archive,'w',compression=zipfile.ZIP_DEFLATED) as zipped:
    for path in RESULTS.glob('*'):
        zipped.write(path,f'results/{path.name}')
    for path in [ROOT/'environment.json', ROOT/'downloads.json']:
        zipped.write(path,path.name)
files.download(str(archive))


In [ ]:
#@title 7. 생성 이미지 보기
CANDIDATE_ID = 'N1-lora8' #@param {type:'string'}
SCENARIO = 'first' #@param ['first','warm_same_prompt','warm_new_prompt','warm_settings']
from IPython.display import Image
image = RESULTS/f'{CANDIDATE_ID}-{SCENARIO}.png'
if image.exists():
    display(Image(filename=str(image)))
else:
    print('해당 후보·시나리오의 생성 결과를 확인하세요.')
